# VFX Production Intelligence Dashboard — Shots Cleaning

Work through this table from top to bottom. The context and rules below are inherited from earlier milestones.

## What we already know

- **Business name:** Shots
- **Purpose:** Contains records for shots used by the approved project analysis, including owning project, editorial sequence identifier, and shot code within a project.
- **One row represents:** One row per shot
- **Expected primary key:** `shot_id`

### Relationships
- project_id → projects.project_id
- assigned_artist_id → artists.artist_id

### Field rules from the approved data dictionary

| Field | Definition | Expected type | Null rule | Key role | Uniqueness | Allowed values / format | Cleaning expectation |
|---|---|---|---|---|---|---|---|
| `shot_id` | Unique identifier combining the owning project, editorial sequence, and shot code. | Identifier text | No nulls allowed | Primary key | Required | PRJ-####_SQ###_SH####. | Remove exact duplicate rows, quarantine conflicting repeated IDs, standardize casing and whitespace, and verify uniqueness and completeness. |
| `project_id` | Identifier of the project that owns the production shot. | Identifier text | No nulls allowed | Foreign key | Not required | PRJ-####. | Trim and uppercase identifiers, validate against projects.project_id, reconcile the project prefix in shot_id, and quarantine unresolved references. |
| `sequence_code` | Editorial sequence identifier within the owning project. | Identifier text | No nulls allowed | Not a key | Not required | SQ###: uppercase SQ followed by three digits. | Trim whitespace, standardize casing, and verify that the value matches the sequence portion of shot_id. |
| `shot_name` | Shot code used to identify the shot within its project. | Identifier text | No nulls allowed | Not a key | Not required | SQ###_SH####. | Trim whitespace, standardize casing, and verify consistency with the sequence and shot portions of shot_id. |
| `department` | Production department responsible for the shot’s current or final task. | Category text | No nulls allowed | Not a key | Not required | Animation; Assets; Compositing; FX; Lighting; Matchmove; Matte Painting; Roto/Paint. | Trim whitespace and map known variants such as Tracking to Matchmove, ROTO to Roto/Paint, and DMP to Matte Painting. |
| `assigned_artist_id` | Identifier of the primary artist assigned to the production shot. | Identifier text | Conditionally nullable | Foreign key | Not required | ART-###, or null only while a shot is legitimately unassigned. | Trim and uppercase IDs, validate against artists.artist_id, preserve null only for legitimately unassigned work, and correct or quarantine missing assignments on progressed shots. |
| `assigned_date` | Date on which the shot was assigned to its primary artist. | Date | Conditionally nullable | Not a key | Not required | YYYY-MM-DD, or null only while the shot is unassigned. | Parse valid formats, preserve null only for genuinely unassigned shots, and validate assigned_date against assignment and start_date. |
| `start_date` | Date production work began on the shot. | Date | Conditionally nullable | Not a key | Not required | YYYY-MM-DD, or null only while status is Not Started. | Parse valid formats, preserve null only for not-started work, and correct or quarantine invalid or missing dates on progressed shots. |
| `deadline` | Required delivery date for the production shot. | Date | No nulls allowed | Not a key | Not required | YYYY-MM-DD and on or after assigned_date/start_date when those dates exist. | Parse valid formats, correct or quarantine the missing value, and validate logical order against assignment, start, and delivery dates. |
| `delivery_date` | Date on which the shot was delivered. | Date | Conditionally nullable | Not a key | Not required | YYYY-MM-DD, or null until the shot is delivered. | Parse valid dates, preserve nulls for undelivered shots, require a date for delivered work, and validate it against start_date and deadline. |
| `first_pass_date` | Date the shot was first submitted for production review. | Date | Conditionally nullable | Not a key | Not required | YYYY-MM-DD, or null before the first review submission. | Parse valid dates, preserve null only before first submission, and quarantine invalid or chronologically inconsistent values. |
| `final_approval_date` | Date the shot received final production approval. | Date | Conditionally nullable | Not a key | Not required | YYYY-MM-DD, or null until final approval. | Parse valid dates, preserve nulls before approval, require a date for approved/final/delivered work as appropriate, and validate chronological order. |
| `status` | Current or final workflow stage of the production shot. | Category text | No nulls allowed | Not a key | Not required | Not Started; WIP; Pending Review; Client Review; Approved; Final; Delivered; On Hold. | Trim whitespace and map known variants to the eight approved workflow statuses. |
| `priority` | Urgency assigned to the production shot for scheduling and resource decisions. | Category text | No nulls allowed | Not a key | Not required | Low; Medium; High; Critical. | Standardize capitalization and investigate or fill missing priority only from an approved source. |
| `complexity` | Estimated production complexity used for planning and workload analysis. | Category text | No nulls allowed | Not a key | Not required | Low; Low-Medium; Medium; High; Very High. | Standardize to the approved complexity scale and investigate or fill missing values only from an approved source. |
| `estimated_hours` | Planned labor hours required to complete the shot. | Decimal | No nulls allowed | Not a key | Not required | Positive numeric value. | Convert valid values to decimal, reject non-positive values, investigate outliers, and quarantine unresolved values from estimate-accuracy calculations. |
| `actual_hours` | Reported total labor hours spent on the shot. | Decimal | Conditionally nullable | Not a key | Not required | Non-negative numeric value; null only before work is recorded. | Convert valid values to decimal, remove text suffixes, preserve null only before work is recorded, reject negatives, investigate zero/outliers, and reconcile against summed time entries. |
| `internal_revision_count` | Number of internal revision rounds recorded for the shot. | Integer | No nulls allowed | Not a key | Not required | Non-negative whole number. | Preserve valid non-negative integers and reconcile against internal review-event records. |
| `client_revision_count` | Number of client-requested revision rounds recorded for the shot. | Integer | No nulls allowed | Not a key | Not required | Non-negative whole number. | Preserve valid non-negative integers and reconcile against client review-event records. |
| `revision_count_reported` | Raw total revision count reported by the production tracking export. | Integer | No nulls allowed | Not a key | Not required | Non-negative whole number equal to the intended reported total. | Reject negative values, investigate missing values, and reconcile the reported total against internal_revision_count, client_revision_count, and review events. |
| `hold_days` | Number of calendar days the shot was blocked or placed on hold. | Integer | No nulls allowed | Not a key | Not required | Non-negative whole number. | Preserve valid non-negative integers and validate consistency with On Hold status and production notes. |
| `vendor_dependencies` | External dependency category that may affect completion of the shot. | Category text | No nulls allowed | Not a key | Not required | None; 3D Asset; Plate Prep; Client Plate; External Simulation. | Trim and standardize categories; investigate blanks rather than treating them as None automatically. |
| `notes` | Optional free-form production notes describing shot-specific context or constraints. | Text | Nulls allowed | Not a key | Not required | Free-form text, or null when no note is recorded. | Trim populated text, convert empty strings to null, and preserve the original wording. |

### Known issues and approved decisions
- shot_id: The raw table contains 1,136 rows and 1,112 distinct shot IDs, producing 24 repeated rows.
- shot_id: shot_id remains the intended primary key. Repeated IDs are raw-data defects rather than a change to the one-shot grain.
- project_id: The raw field contains 18 project-key issues, including whitespace, lowercase IDs, and nonexistent PRJ-9999 values.
- project_id: project_id remains a required foreign key. Recoverable formatting issues will be standardized; nonexistent or contradictory values require correction or quarantine.
- department: The raw field contains 16 category issues, including trailing spaces, case variants, Tracking, ROTO, and DMP.
- department: The raw variants represent the approved department categories and will be standardized before departmental reporting.
- assigned_artist_id: The raw field contains six blanks and additional whitespace or nonexistent ART-999 references; some blanks occur on completed shots.
- assigned_artist_id: Null is permitted only before assignment. Populated IDs remain foreign keys, and missing or orphaned assignments on progressed shots require remediation.
- assigned_date: The raw text field contains blanks, TBD, timestamps, and alternate date formats; some missing values occur on completed shots.
- assigned_date: The field is logically a date; mixed formats and inappropriate blanks explain the text type and require cleaning.
- start_date: The raw text field contains mixed formats, TBD, and 23 blanks; some blanks occur on progressed shots.
- start_date: The field is logically a date. Conditional nulls are valid before work starts; other blanks and malformed values require remediation.
- deadline: The raw text field includes alternate formats, one blank, and date-order defects.
- deadline: The field is a required date; formatting and date-order defects must be resolved before overdue and on-time-delivery analysis.
- delivery_date: The raw text field contains 154 blanks plus alternate formats and invalid sequencing; some nulls are expected for open shots.
- delivery_date: Nulls are valid only for undelivered shots; mixed formats and impossible date order require cleaning.
- first_pass_date: The raw text field includes blanks, mixed formats, and invalid dates such as 2025-13-05.
- first_pass_date: The field is logically a conditional date; malformed values and inappropriate blanks require remediation.
- final_approval_date: The raw text field includes 156 blanks, TBD values, and alternate formats; many blanks are expected for work not yet approved.
- final_approval_date: Nulls are conditional on workflow status; mixed or invalid source values will be standardized or quarantined.
- status: The raw field contains 24 inconsistencies such as W.I.P., In Progress, Pend Review, On-Hold, case variants, and trailing whitespace.
- status: The source uses multiple labels for the same shot states; standardization is required before status-based KPI analysis.
- priority: The raw field contains four blanks.
- priority: Priority is required for planning; the raw blanks are incomplete descriptive values that need confirmation.
- complexity: The raw field contains three blanks.
- complexity: Complexity is expected for planning; raw blanks require confirmation rather than silent imputation.
- estimated_hours: The raw text field contains N/A, blanks, zero, negative values, and extreme outliers including 1200.
- estimated_hours: The field is expected to be positive numeric; invalid source strings, missing values, non-positive values, and outliers explain the text type and require remediation.
- actual_hours: The raw text field contains pending, 18h, blanks, zero, negative values, and extreme outliers including 950.5; some blanks occur on completed work.
- actual_hours: The field is logically numeric and conditionally nullable; malformed, non-positive, missing-on-completed, and extreme values require cleaning and reconciliation.
- revision_count_reported: The raw field contains 11 blanks, 14 values of -1, and 42 reconciliation issues against component counts or review events.
- revision_count_reported: The reported count is expected to be a complete non-negative integer; missing, negative, or inconsistent values require reconciliation.
- vendor_dependencies: The raw field contains five blanks; None is the approved value when no dependency exists.
- vendor_dependencies: A dependency status is required. Blank values are incomplete because no dependency should be represented explicitly as None.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r'C:/Users/Dan/Documents/MEGA/Dev/GitHub/career-accelerator/projects/project-01-vfx-production-intelligence')
TABLE_NAME = 'shots'
RAW_PATH = PROJECT_DIR / r'data/raw/csv/raw_shots.csv'
PROCESSED_PATH = PROJECT_DIR / r'data/processed/csv/shots.csv'
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

if RAW_PATH.suffix.lower() == '.csv':
    raw_df = pd.read_csv(RAW_PATH)
elif RAW_PATH.suffix.lower() == '.parquet':
    raw_df = pd.read_parquet(RAW_PATH)
else:
    raise ValueError(f'Add the appropriate pandas reader for {RAW_PATH.suffix}')

clean_df = raw_df.copy()
print(f'{TABLE_NAME}: {len(raw_df):,} raw rows, {len(raw_df.columns)} columns')
raw_df.head()


## 1. Profile the raw table

- Confirm the source row count and column names.
- Measure missing values by field.
- Check exact duplicate rows.
- Test uniqueness and nulls for `shot_id`.
- Review observed categories and parsing problems.
- Compare findings with the dictionary rules above before changing data.

In [ ]:
# Write the profiling checks for this table here.
# Keep the outputs that justify your cleaning decisions.


## 2. Apply the approved cleaning plan

Transform `clean_df` without modifying `raw_df`. Follow the field-level expectations above. Document any treatment that differs from the approved dictionary.

In [ ]:
# Write this table's cleaning transformations here.
# Example structure only: clean_df = clean_df.copy()


## 3. Validate the processed result

- Required columns are still present.
- Expected logical types can be produced consistently.
- Required fields do not contain unresolved nulls.
- Allowed values and formats match the dictionary.
- Invalid negative, out-of-range, or impossible values are resolved or documented.
- `shot_id` is non-null and unique.
- Foreign-key and relationship exceptions are measured and documented.

In [ ]:
# Write the before-and-after validation checks here.
# The checks should fail visibly when an unresolved issue remains.


## 4. Export the reviewed table

After validation, save the reviewed result to `data/processed/csv/shots.csv`. The Data Cleaning Studio will discover and validate the file.

In [ ]:
# Run only after the table has passed your validation checks.
clean_df.to_csv(PROCESSED_PATH, index=False)
print(f'Saved {len(clean_df):,} rows to {PROCESSED_PATH}')


## Cleaning summary

<!-- Describe what changed, why each important decision was appropriate, how many records were affected, and any remaining exception that a later milestone must know about. -->
